# Organize SnFR stimulus-duration CSVs into HDF5

This notebook reads the neurotransmitter-sensor (iAChSnFR, iGluSnFR) recordings and writes them into one HDF5 file.

**Expected folder layout**
```
DATA_ROOT/                              e.g. snfr
    <experiment>/                       iachsnfr, iglusnfr
        <condition>/                    e.g. CTR, TNT, PFKRi, AMPKRi
            <compartment>/              e.g. M9, M5
                260202_s15_6M9_iglusnfr_mean_stim0p49s.csv      trace file
                260202_s15_6M9_ca_mean_stim0p49s.csv
                260202_s15_stim0p49s_frequency.csv              frequency file
```
- **Trace files** are named `<date>_s<series>_<ROI label>_<sensor>_mean_stim<duration>s.csv`. The sensor is the part between the ROI label (`6M9`) and `_mean`, for example `iglusnfr`, `iachsnfr2` or `ca`. The duration uses `p` for the decimal point (`stim0p49s` = 0.49 s). Rows are neurons and columns are time points, with no header row (if the first row is mostly text, it is used as a header).
- **Frequency files** (`<date>_s<series>_stim<duration>s_frequency.csv`) are matched to the trace files of the same fly and duration. If a compartment folder has no frequency file for a recording, a frequency file for the same fly and duration in another compartment folder of the same condition is used, provided they all give the same value.

**Output:** one HDF5 file laid out as `/<experiment>/<fly_id>/<compartment>/<sensor>/<stimulus>`, for example `/iglusnfr/260202_s15/M9/iglusnfr/stim0p49s`. Each dataset is neurons × time points and carries `stim_time`, `imaging_frequency`, `roi_label` and the source file; the condition is stored on each fly. The experiment level keeps the `ca` recordings of the two experiments apart.

**How to use:** check the paths in section 1, run section 3 (dry run) and look over the output, then run section 4 to write the file.

In [ ]:
from collections import defaultdict
from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd

## 1. Paths and settings

In [ ]:
DATA_ROOT = Path("your/path/here")   # folder containing iachsnfr and iglusnfr
OUT_FILE = Path("your/path/here/snfr.h5")   # the HDF5 file to write

# Trace file name: <date>_s<series>[_<ROI label>]_<sensor>_mean_stim<duration>s.csv
TRACE_PATTERN = re.compile(
    r"^(?P<date>\d{6})_s(?P<series>\d+)(?:_(?P<roi>\d+[^_]*))?_(?P<sensor>.+?)"
    r"_mean_stim(?P<stim>[0-9p]+)s$",
    re.IGNORECASE,
)

# Frequency file name: <date>_s<series>[_stim<duration>s]_frequency.csv
FREQUENCY_PATTERN = re.compile(
    r"^(?P<date>\d{6})_s(?P<series>\d+)(?:_stim(?P<stim>[0-9p]+)s)?_freq(?:uency)?$",
    re.IGNORECASE,
)

# Compression for the HDF5 datasets (None to disable).
COMPRESSION = "gzip"

## 2. Functions
Run these cells; no changes needed.

In [ ]:
def stim_label_to_float(label):
    """0p49 -> 0.49"""
    return float(label.lower().replace("p", "."))


def parse_file(csv_path, data_root):
    """Return (metadata dict, None) for one CSV, or (None, reason) if parsing fails."""
    rel = csv_path.relative_to(data_root)
    folders = list(rel.parts[:-1])
    if len(folders) != 3:
        return None, "not inside an <experiment>/<condition>/<compartment>/ folder"
    experiment, condition, compartment = folders
    stem = csv_path.stem

    rec = {
        "path": csv_path,
        "rel": str(rel),
        "experiment": experiment,
        "condition": condition,
        "compartment": compartment,
    }

    m = FREQUENCY_PATTERN.match(stem)
    if m:
        rec.update(kind="frequency", date=m["date"], series=m["series"],
                   stim_label=(m["stim"] or "").lower())
    else:
        m = TRACE_PATTERN.match(stem)
        if not m:
            return None, ("name does not match <date>_s<series>_<ROI>_<sensor>_mean_stim<duration>s "
                          "or <date>_s<series>_stim<duration>s_frequency")
        rec.update(kind="trace", date=m["date"], series=m["series"], roi_label=m["roi"] or "",
                   sensor=m["sensor"].lower(), stim_label=m["stim"].lower())
        rec["stim_time"] = stim_label_to_float(rec["stim_label"])

    rec["fly_id"] = f"{rec['date']}_s{rec['series']}"
    return rec, None

In [ ]:
def read_frequency(csv_path):
    """Return the first numeric value found in a frequency CSV, or None."""
    df = pd.read_csv(csv_path, header=None, dtype=str)
    for value in df.to_numpy().ravel():
        try:
            return float(str(value).strip())
        except ValueError:
            continue
    return None


def _row_is_numeric_like(row):
    vals = pd.to_numeric(row, errors="coerce")
    return np.isfinite(vals).sum() >= max(1, len(row) // 2)


def read_traces(csv_path):
    """Return traces as an array [neurons x time points], read with header=None."""
    df = pd.read_csv(csv_path, header=None)
    if not df.empty and not _row_is_numeric_like(df.iloc[0]):
        df = pd.read_csv(csv_path, header=0)
    return df.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float64)

In [ ]:
def keep_order(rec):
    """Order in which duplicate recordings are considered (by path); the first
    one is kept."""
    return rec["rel"]


def place(r):
    """Where a recording belongs: experiment, condition, compartment, fly."""
    return (r["experiment"], r["condition"], r["compartment"], r["fly_id"])


def collect(data_root):
    data_root = Path(data_root)
    traces, freq_files, problems, notes = [], [], [], []
    for csv_path in sorted(data_root.rglob("*.csv")):
        rec, reason = parse_file(csv_path, data_root)
        if rec is None:
            problems.append((str(csv_path.relative_to(data_root)), reason))
        elif rec["kind"] == "frequency":
            freq_files.append(rec)
        else:
            traces.append(rec)

    # Duplicate trace files: same fly, compartment, sensor and duration twice.
    # Only the first in keep order (see keep_order) is written.
    seen = defaultdict(list)
    for rec in sorted(traces, key=keep_order):
        seen[place(rec) + (rec["sensor"], rec["stim_label"])].append(rec["rel"])
    for key, rels in seen.items():
        if len(rels) > 1:
            problems.append((rels[0], f"duplicate: same fly, compartment, sensor and duration; "
                                      f"keeping {rels[0]}, leaving out {rels[1:]}"))

    # Frequency values: per place + duration, with "" meaning "all durations".
    freq_values = {}
    freq_by_fly = defaultdict(list)   # (experiment, condition, fly, duration) -> [(value, file)]
    for rec in freq_files:
        value = read_frequency(rec["path"])
        if value is None:
            problems.append((rec["rel"], "no numeric value found in frequency file"))
            continue
        freq_values[place(rec) + (rec["stim_label"],)] = (value, rec["rel"])
        freq_by_fly[(rec["experiment"], rec["condition"], rec["fly_id"],
                     rec["stim_label"])].append((value, rec["rel"]))

    used = set()
    for rec in traces:
        match = (freq_values.get(place(rec) + (rec["stim_label"],))
                 or freq_values.get(place(rec) + ("",)))
        if match is None:
            # Fall back to the same fly in another compartment folder of the same
            # condition, if all such frequency files agree.
            for stim in (rec["stim_label"], ""):
                candidates = freq_by_fly.get((rec["experiment"], rec["condition"],
                                              rec["fly_id"], stim), [])
                values = {v for v, _ in candidates}
                if len(values) == 1:
                    match = candidates[0]
                    break
                if len(values) > 1:
                    problems.append((rec["rel"], f"no frequency file in this folder, and the ones "
                                                 f"in other folders disagree: {[r for _, r in candidates]}"))
                    break
        rec["frequency"], rec["frequency_file"] = match if match else (np.nan, "")
        if match:
            used.add(match[1])

    # Frequency files that no trace file needed. Not an error: usually the
    # fly has no trace files for that duration in that folder.
    for rec in freq_files:
        if rec["rel"] not in used and any(rec["rel"] == v[1] for v in freq_values.values()):
            notes.append((rec["rel"], "frequency file not used: no trace files for this fly and duration in this folder"))

    # Within an experiment, a fly ID may belong to only one condition.
    conditions_per_fly = defaultdict(set)
    for rec in traces:
        conditions_per_fly[(rec["experiment"], rec["fly_id"])].add(rec["condition"])
    clashes = {key: c for key, c in conditions_per_fly.items() if len(c) > 1}
    for (exp, fly), conds in sorted(clashes.items()):
        problems.append((f"{exp}/{fly}", f"same fly ID under several conditions {sorted(conds)}; "
                                         "not written"))
    traces = [rec for rec in traces if (rec["experiment"], rec["fly_id"]) not in clashes]

    # Neuron counts from the file contents; they should not change across durations.
    for rec in traces:
        rec["n_rows"] = read_traces(rec["path"]).shape[0]
    rows_per_sensor = defaultdict(set)
    for rec in traces:
        rows_per_sensor[place(rec) + (rec["sensor"],)].add(rec["n_rows"])
    for (exp, cond, comp, fly, sensor), counts in sorted(rows_per_sensor.items()):
        if len(counts) > 1:
            notes.append((f"{exp}/{cond}/{comp}/{fly}/{sensor}",
                          f"different numbers of rows across durations: {sorted(counts)}"))

    return traces, problems, notes


def print_summary(records, problems, notes):
    print(f"Parsed {len(records)} trace files.\n")
    if records:
        df = pd.DataFrame(records)
        table = (df.groupby(["experiment", "condition", "compartment", "sensor", "stim_time"])
                   .agg(n_flies=("fly_id", "nunique"), n_neurons=("n_rows", "sum"))
                   .reset_index())
        print("Flies and neurons per experiment / condition / compartment / sensor / duration:")
        print(table.to_string(index=False))

    no_freq = sorted({r["rel"] for r in records if np.isnan(r["frequency"])})
    if no_freq:
        print("\nTrace files without a frequency file:")
        for rel in no_freq:
            print(f"  {rel}")

    if notes:
        print(f"\n{len(notes)} note(s) (no action needed unless a neuron count is listed):")
        for where, text in notes:
            print(f"  {where}: {text}")

    if problems:
        print(f"\n{len(problems)} problem(s):")
        for rel, reason in problems:
            print(f"  {rel}: {reason}")

In [ ]:
def write_hdf5(records, out_path):
    """Write all records into one HDF5 file at out_path."""
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    with h5py.File(out_path, "w") as f:
        for r in sorted(records, key=keep_order):
            name = f"stim{r['stim_label']}s"
            path_in_file = (f"{r['experiment']}/{r['fly_id']}/{r['compartment']}/"
                            f"{r['sensor']}/{name}")
            if path_in_file in f:
                # A duplicate (reported in the dry run): leave the file out and
                # change nothing that was already written.
                print(f"  skipping duplicate /{path_in_file}: {r['rel']}")
                continue

            exp = f.require_group(r["experiment"])
            exp.attrs["experiment"] = r["experiment"]

            fly = exp.require_group(r["fly_id"])
            fly.attrs["condition"] = r["condition"]
            fly.attrs["date"] = r["date"]
            fly.attrs["series"] = r["series"]

            sensor = fly.require_group(r["compartment"]).require_group(r["sensor"])
            dset = sensor.create_dataset(name, data=read_traces(r["path"]),
                                         compression=COMPRESSION)
            for key in ("experiment", "condition", "compartment", "date", "series", "sensor",
                        "stim_label", "stim_time", "roi_label", "frequency_file"):
                dset.attrs[key] = r[key]
            dset.attrs["imaging_frequency"] = r["frequency"]
            dset.attrs["source_file"] = str(r["path"])
            dset.attrs["dims"] = "neurons x time"
            written += 1
    n_flies = len({(r["experiment"], r["fly_id"]) for r in records})
    n_experiments = len({r["experiment"] for r in records})
    print(f"Wrote {out_path} ({written} recordings, {n_flies} flies, {n_experiments} experiments)")

## 3. Dry run
Shows how the files were parsed, without writing anything. Check the table of flies and neurons, files without a frequency file, the notes, and the list of problems.

In [ ]:
records, problems, notes = collect(DATA_ROOT)
print_summary(records, problems, notes)

Optional: every parsed file as a table, for sorting or filtering.

In [ ]:
pd.DataFrame([{k: v for k, v in r.items() if k not in ("kind", "path")} for r in records])

## 4. Write the HDF5 file
This overwrites `OUT_FILE` if it already exists.

In [ ]:
write_hdf5(records, OUT_FILE)

## 5. Look at the file (example)

In [ ]:
with h5py.File(OUT_FILE, "r") as f:
    for experiment, exp in f.items():
        print(experiment)
        for fly_id, fly in exp.items():
            print(f"  {fly_id}  condition: {fly.attrs['condition']}")
            for compartment, comp in fly.items():
                for sensor, sgrp in comp.items():
                    stims = ", ".join(f"{d.attrs['stim_time']:g} s" for d in sgrp.values())
                    n = next(iter(sgrp.values())).shape[0]
                    print(f"      {compartment}/{sensor}: {n} neurons; durations {stims}")